[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Smiledxd/python_data/blob/main/sesiones/S13_pandas_integrador.ipynb)

# Sesión 13 · Integrador de pandas: un caso bancario de principio a fin

**Módulo 3: Pandas** · ⏱️ Duración estimada: 60 minutos (más el avance del proyecto, que es trabajo aparte)

## 🎯 Objetivos
Al terminar esta sesión podrás:
1. Cargar un caso con varios archivos y apilarlos.
2. Limpiar textos, tipos, fechas, duplicados, centinelas y "unknown" en tablas reales.
3. Unir tablas sin perder filas y resumir con `groupby`, `crosstab`, `value_counts` y `resample`.
4. Comparar grupos de distinto tamaño con una métrica **normalizada** (reclamos por cada 100 clientes).

## 📋 Qué debes saber antes
Sesiones 9 a 12. Esta sesión no trae conceptos nuevos: cada etapa recuerda en pocas líneas lo que vas a usar.

## 🧭 El caso
Eres analista de un banco. Te entregan cinco archivos: la base de clientes, las transacciones de enero a marzo (un archivo por mes) y los reclamos del trimestre. Gerencia pregunta: **¿en qué distritos tenemos un problema de reclamos?** Para responder bien hay que limpiar, unir y, sobre todo, comparar de forma justa.

## 🧭 Cómo trabajar este notebook
- Ejecuta las celdas **en orden**, de arriba abajo, con **Shift + Enter**.
- Cada etapa usa lo que dejó la anterior. Si una verificación falla, corrígela antes de seguir.
- Si te atascas, abre la 💡 **Pista**. Hay dos, de menos a más ayuda.

## ⚙️ Setup
Ejecuta la celda siguiente **al empezar** (y otra vez si reinicias el entorno). Escribe los cinco archivos del caso en la carpeta de trabajo y carga las funciones que revisan tus respuestas.

⚠️ **Ejecútala y no la edites.**

In [ ]:
#@title ⚙️ Setup: ejecuta esta celda y no la edites { display-mode: "form" }
# Escribe los archivos del caso (datos de práctica generados aquí mismo) y carga los verificadores.
import copy
import csv
import datetime as dt
import hashlib
import io
import math
import statistics
from collections import defaultdict

import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

# ---------- clientes_banco.csv ----------
_TAM = {"Surco": 20, "Miraflores": 15, "San Isidro": 12, "Lince": 8, "Barranco": 5}
_LAMBDA = {"Surco": 0.6, "Miraflores": 0.4, "San Isidro": 0.3, "Lince": 0.5, "Barranco": 1.4}   # reclamos esperados por cliente
_distritos = [d for d, n in _TAM.items() for _ in range(n)]
rng.shuffle(_distritos)
_COLS_CLI = ["cliente_id", "fecha_alta", "distrito", "edad", "segmento", "ingreso_mensual"]
_cli = []
for _k, _d in enumerate(_distritos):
    _var = [_d, _d.upper(), _d.lower(), f" {_d}  "][int(rng.integers(0, 4))]
    _edad = -1 if _k in (7, 22, 41) else int(rng.integers(20, 75))
    _seg = "unknown" if _k in (3, 18, 33, 50) else str(rng.choice(["clásico", "preferente", "premium"], p=[0.5, 0.3, 0.2]))
    _cli.append([f"B{_k + 1:03d}", f"{int(rng.integers(1, 29)):02d}/{int(rng.integers(1, 13)):02d}/20{int(rng.integers(18, 26))}",
                 _var, _edad, _seg, round(float(rng.uniform(1200, 12000)), 2)])


def _escribir(nombre, cols, filas, sep=","):
    buf = io.StringIO()
    csv.writer(buf, delimiter=sep, lineterminator="\n").writerows([cols] + filas)
    with open(nombre, "w", encoding="utf-8") as f:
        f.write(buf.getvalue())
    return buf.getvalue()


_TXT = {"clientes_banco.csv": _escribir("clientes_banco.csv", _COLS_CLI, _cli)}

# ---------- transacciones_2026_01.csv, _02.csv y _03.csv (separadas por punto y coma) ----------
_COLS_TX = ["id_tx", "fecha", "cliente_id", "tipo", "canal", "monto", "estado"]
_ids = [f"B{k:03d}" for k in range(1, 61)]
_meses, _n_tx = {}, 90001
for _mes in (1, 2, 3):
    _filas = []
    for _ in range(70):
        _cl = "B999" if rng.random() < 0.03 else str(rng.choice(_ids))
        _t = str(rng.choice(["deposito", "retiro", "pago", "transferencia"], p=[0.25, 0.35, 0.25, 0.15]))
        _tipo = [_t, _t.upper(), f" {_t.title()}"][int(rng.integers(0, 3))]
        _canal = str(rng.choice(["app", "agencia", "cajero"])) if _t == "retiro" else str(rng.choice(["app", "agencia"]))
        _m = rng.uniform(300, 6000) if _t == "deposito" else -rng.uniform(20, 2500)
        _estado = "rechazada" if rng.random() < 0.08 else "aprobada"
        _filas.append([_n_tx, f"2026-{_mes:02d}-{int(rng.integers(1, 29)):02d}", _cl, _tipo, _canal, f"S/ {_m:,.2f}", _estado])
        _n_tx += 1
    _meses[_mes] = _filas
_meses[2] = [list(f) for f in _meses[1][-2:]] + _meses[2]       # filas repetidas entre archivos
_meses[3].append(list(_meses[3][10]))
for _mes, _filas in _meses.items():
    _TXT[f"transacciones_2026_{_mes:02d}.csv"] = _escribir(f"transacciones_2026_{_mes:02d}.csv", _COLS_TX, _filas, sep=";")

# ---------- reclamos.csv ----------
_MOTIVOS = ["cobro indebido", "demora en atención", "operación no reconocida", "tarjeta bloqueada", "información incorrecta"]
_COLS_REC = ["id_reclamo", "cliente_id", "fecha", "motivo", "resuelto"]
_rec, _n_rec = [], 1
for _k, _d in enumerate(_distritos):
    for _ in range(int(rng.poisson(_LAMBDA[_d]))):
        _mot = str(rng.choice(_MOTIVOS, p=[0.35, 0.25, 0.2, 0.12, 0.08]))
        _rec.append([f"R{_n_rec:03d}", f"B{_k + 1:03d}", f"2026-{int(rng.integers(1, 4)):02d}-{int(rng.integers(1, 29)):02d}",
                     _mot, "Sí" if rng.random() < {"cobro indebido": 0.5, "demora en atención": 0.8}.get(_mot, 0.65) else "No"])
        _n_rec += 1
_TXT["reclamos.csv"] = _escribir("reclamos.csv", _COLS_REC, _rec)

# ---------- Herramientas de verificación ----------
_FALTA = object()


def _h(valor):
    if isinstance(valor, str):
        valor = valor.strip().lower()
    return hashlib.sha256(f"{type(valor).__name__}|{valor!r}".encode("utf-8")).hexdigest()


def _cerca(a, b, tol=1e-9):
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=tol)


def _dos_decimales(x):
    return abs(x * 100 - round(x * 100)) < 1e-6


def _corto(valor, n=60):
    if type(valor).__module__ == "numpy" and getattr(valor, "shape", None) == ():
        valor = valor.item()
    texto = repr(valor)
    return texto if len(texto) <= n else texto[:n] + "…"


def _igual(a, b, tol=1e-6):
    """Compara exigiendo el mismo tipo en None/bool y tolerancia en decimales."""
    if b is None or isinstance(b, bool):
        return type(a) is type(b) and a == b
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return (isinstance(a, (int, float)) and not isinstance(a, bool)
                and math.isclose(a, b, rel_tol=1e-9, abs_tol=tol))
    if isinstance(b, (list, tuple)):
        return (type(a) is type(b) and len(a) == len(b)
                and all(_igual(x, y, tol) for x, y in zip(a, b)))
    if isinstance(b, dict):
        return (isinstance(a, dict) and set(a) == set(b)
                and all(_igual(a[k], b[k], tol) for k in b))
    return type(a) is type(b) and a == b


class _Revision:
    def __init__(self, titulo):
        self.titulo = titulo
        self.errores = 0
        print(f"── {titulo} ──")

    def ok(self, msg):
        print(f"✅ {msg}")

    def mal(self, msg):
        self.errores += 1
        print(f"❌ {msg}")

    def var(self, nombre, tipo=None):
        valor = globals().get(nombre, _FALTA)
        if valor is _FALTA:
            self.mal(f"No encuentro `{nombre}`. ¿Ejecutaste tu celda? ¿Escribiste bien el nombre?")
            return _FALTA
        if tipo is not None and not (type(valor) is tipo or (isinstance(tipo, tuple) and type(valor) in tipo)):
            esperado = tipo.__name__ if not isinstance(tipo, tuple) else " o ".join(t.__name__ for t in tipo)
            self.mal(f"`{nombre}` es de tipo {type(valor).__name__} y se esperaba {esperado}.")
            return _FALTA
        return valor

    def funcion(self, nombre):
        f = self.var(nombre)
        if f is _FALTA:
            return _FALTA
        if not callable(f):
            self.mal(f"`{nombre}` existe pero no es una función. ¿La definiste con `def`?")
            return _FALTA
        return f

    def caso(self, texto, f, args=(), kwargs=None, esperado=None, igual=None, motivo="no es lo esperado", tol=0.0051):
        """Llama a f con copias de los argumentos y compara sin mostrar el valor esperado."""
        import copy
        try:
            obtenido = f(*copy.deepcopy(args), **copy.deepcopy(kwargs or {}))
        except Exception as e:
            self.mal(f"`{texto}` lanzó {type(e).__name__}: {e}")
            return False
        bien = igual(obtenido, esperado) if igual else _igual(obtenido, esperado, tol)
        if bien:
            self.ok(f"`{texto}` funciona.")
        else:
            self.mal(f"`{texto}` devolvió {_corto(obtenido)}; {motivo}.")
        return bien

    def valor(self, nombre, esperado, tipo=None, pista="revisa el cálculo", igual=None):
        v = self.var(nombre, tipo)
        if v is _FALTA:
            return
        bien = igual(v, esperado) if igual else _igual(v, esperado)
        if bien:
            self.ok(f"`{nombre}` es correcto.")
        else:
            self.mal(f"`{nombre}` vale {_corto(v)}; {pista}.")

    def texto_limpio(self, nombre, valor):
        if valor != valor.strip():
            self.mal(f"`{nombre}` tiene espacios o saltos de línea al inicio o al final: {valor!r}")
            return False
        return True

    def predicciones(self, esperados):
        for nombre, hash_ok in esperados.items():
            v = self.var(nombre)
            if v is _FALTA:
                continue
            if _h(v) == hash_ok:
                self.ok(f"`{nombre}` es correcto.")
            else:
                self.mal(f"`{nombre}` no es correcto. Razónalo otra vez y luego compruébalo ejecutando la expresión en una celda nueva.")

    def fin(self):
        if self.errores == 0:
            print(f"🎉 ¡{self.titulo} superado!")
        else:
            cuantos = "el punto marcado" if self.errores == 1 else f"los {self.errores} puntos marcados"
            print(f"🔁 Corrige {cuantos} con ❌ y vuelve a verificar.")


def _primera_diferencia(r, nombre, tuyo, esperado):
    for i, (a, b) in enumerate(zip(tuyo, esperado)):
        if a != b:
            r.mal(f"`{nombre}` no tiene el formato pedido. La diferencia empieza en el carácter {i}: "
                  f"desde ahí tu texto dice {tuyo[i:i + 15]!r}.")
            return
    n = abs(len(esperado) - len(tuyo))
    cuantos = "1 carácter" if n == 1 else f"{n} caracteres"
    if len(tuyo) < len(esperado):
        r.mal(f"`{nombre}` está incompleto: le {'falta' if n == 1 else 'faltan'} {cuantos} al final.")
    else:
        r.mal(f"`{nombre}` tiene {cuantos} de más al final: {tuyo[len(esperado):]!r}.")


def _es_numero(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, (bool, np.bool_))


def _esc(r, nombre, esperado, pista, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if isinstance(v, np.ndarray) and v.shape == ():
        v = v.item()
    if not _es_numero(v):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un número.")
    elif abs(float(v) - esperado) <= tol:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` vale {_corto(v.item() if hasattr(v, 'item') else v)}; {pista}.")


def _arr(r, nombre, esperado, pista, tol=1e-6, tipos=None):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, np.ndarray):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un array de NumPy (`np.ndarray`).")
        return
    esperado = np.array(esperado)
    if v.shape != esperado.shape:
        r.mal(f"`{nombre}` tiene forma {v.shape} y se esperaba {esperado.shape}.")
        return
    if tipos and v.dtype.kind not in tipos:
        nombres = {"b": "bool", "i": "entero", "u": "entero", "f": "decimal (float)", "U": "texto"}
        r.mal(f"`{nombre}` tiene dtype {v.dtype} y se esperaba un tipo {' o '.join(sorted({nombres[t] for t in tipos}))}.")
        return
    if v.dtype.kind in "USO" or esperado.dtype.kind in "USO":
        bien = v.tolist() == esperado.tolist()
    elif v.dtype.kind == "b" or esperado.dtype.kind == "b":
        bien = np.array_equal(v, esperado)
    else:
        bien = np.allclose(v.astype(float), esperado.astype(float), rtol=0, atol=tol, equal_nan=True)
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")




def _sin_cambios(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        original = _D[n]
        if isinstance(original, np.ndarray):
            igual = isinstance(actual, np.ndarray) and actual.shape == original.shape and np.array_equal(actual, original, equal_nan=original.dtype.kind == "f")
        else:
            igual = actual == original
        if not igual:
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a ejecutar el setup.")


def _norm(x):
    if isinstance(x, np.generic):
        x = x.item()
    try:
        if pd.isna(x):
            return None
    except (TypeError, ValueError):
        pass
    if isinstance(x, pd.Timestamp):
        return str(x)
    return x


def _mismo(a, b, tol=1e-6):
    a, b = _norm(a), _norm(b)
    if a is None or b is None:
        return a is None and b is None
    if isinstance(b, (int, float)) and not isinstance(b, bool):
        return isinstance(a, (int, float)) and not isinstance(a, bool) and abs(a - b) <= tol
    return str(a) == str(b)


def _ser(r, nombre, valores, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.Series):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba una Series de pandas.")
        return
    if len(v) != len(valores):
        r.mal(f"`{nombre}` tiene {len(v)} elementos y se esperaban {len(valores)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    if all(_mismo(a, b, tol) for a, b in zip(v.tolist(), valores)):
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene el largo correcto pero sus valores no coinciden; {pista}.")


def _df(r, nombre, columnas, filas, pista, indice=None, tol=1e-6):
    v = r.var(nombre)
    if v is _FALTA:
        return
    if not isinstance(v, pd.DataFrame):
        r.mal(f"`{nombre}` es de tipo {type(v).__name__} y se esperaba un DataFrame de pandas.")
        return
    cols = [str(c) for c in v.columns]
    if cols != columnas:
        faltan = [c for c in columnas if c not in cols]
        sobran = [c for c in cols if c not in columnas]
        if faltan or sobran:
            r.mal(f"A `{nombre}` le faltan las columnas {faltan} y le sobran {sobran}." if faltan and sobran else
                  (f"A `{nombre}` le faltan las columnas {faltan}." if faltan else f"En `{nombre}` sobran las columnas {sobran}."))
        else:
            r.mal(f"`{nombre}` tiene las columnas correctas pero en otro orden.")
        return
    if len(v) != len(filas):
        r.mal(f"`{nombre}` tiene {len(v)} filas y se esperaban {len(filas)}.")
        return
    if indice is not None and [_norm(i) for i in v.index] != indice:
        r.mal(f"Las etiquetas de fila (índice) de `{nombre}` no son las esperadas; {pista}.")
        return
    bien = all(_mismo(a, b, tol) for fila_v, fila_e in zip(v.itertuples(index=False), filas) for a, b in zip(fila_v, fila_e))
    if bien:
        r.ok(f"`{nombre}` es correcto.")
    else:
        r.mal(f"`{nombre}` tiene la forma correcta pero sus valores no coinciden; {pista}.")


def _sin_cambios_df(r, *nombres):
    for n in nombres:
        actual = globals().get(n)
        if not (isinstance(actual, (pd.DataFrame, pd.Series)) and actual.equals(_D[n])):
            r.mal(f"`{n}` cambió. No modifiques los datos originales; vuelve a cargarlos o ejecuta de nuevo el setup.")


def _leer(txt, sep=","):
    """Referencia con el módulo csv (no con pandas), con números convertidos."""
    filas = list(csv.reader(io.StringIO(txt), delimiter=sep))
    salida = []
    for fila in filas[1:]:
        conv = []
        for x in fila:
            try:
                conv.append(int(x))
            except ValueError:
                try:
                    conv.append(float(x))
                except ValueError:
                    conv.append(x)
        salida.append(conv)
    return filas[0], salida


def _ts(d):
    return None if d is None else str(dt.datetime(d.year, d.month, d.day))


_CC, _CLI_RAW = _leer(_TXT["clientes_banco.csv"])
_CT, _TX_RAW = _leer("".join(_TXT[f"transacciones_2026_{m:02d}.csv"].split("\n", 1)[1] if m > 1 else _TXT[f"transacciones_2026_{m:02d}.csv"]
                             for m in (1, 2, 3)), ";")
_CR, _REC_RAW = _leer(_TXT["reclamos.csv"])


def _clientes():
    salida = []
    for f in _CLI_RAW:
        d, m, a = map(int, f[1].split("/"))
        salida.append([f[0], _ts(dt.date(a, m, d)), f[2].strip().title(), None if f[3] == -1 else f[3],
                       None if f[4] == "unknown" else f[4], f[5]])
    return salida


def _tx():
    vistas, salida, dup, rech = set(), [], 0, 0
    for f in _TX_RAW:
        if tuple(f) in vistas:
            dup += 1
            continue
        vistas.add(tuple(f))
        if f[6] != "aprobada":
            rech += 1
            continue
        monto = float(f[5].replace("S/", "").replace(",", "").strip())
        salida.append([f[0], _ts(dt.date.fromisoformat(f[1])), f[2], f[3].strip().lower(), f[4], monto, f[6]])
    return salida, dup, rech


def _reclamos():
    return [[f[0], f[1], _ts(dt.date.fromisoformat(f[2])), f[3], f[4] == "Sí"] for f in _REC_RAW]


def check_etapa_1():
    r = _Revision("Etapa 1 · Cargar")
    _df(r, "clientes_raw", _CC, _CLI_RAW, "carga `clientes_banco.csv` tal cual")
    _df(r, "tx_raw", _CT, _TX_RAW, "carga los tres archivos de transacciones (separados por punto y coma), en orden, con un índice nuevo",
        indice=list(range(len(_TX_RAW))))
    _df(r, "reclamos_raw", _CR, _REC_RAW, "carga `reclamos.csv` tal cual")
    r.valor("forma_tx", (len(_TX_RAW), len(_CT)), tuple, "la forma de `tx_raw`")
    r.fin()


def check_etapa_2():
    r = _Revision("Etapa 2 · Limpiar clientes")
    _df(r, "clientes", _CC, _clientes(), "revisa los cuatro arreglos: fecha día/mes/año, distrito, centinela de edad y \"unknown\"")
    x = r.var("n_distritos")
    if x is not _FALTA:
        r.ok("`n_distritos` es correcto.") if _es_numero(x) and int(x) == len({f[2] for f in _clientes()}) else \
            r.mal(f"`n_distritos` vale {_corto(x)}; cuenta los distritos distintos después de limpiar.")
    r.fin()


def check_etapa_3():
    r = _Revision("Etapa 3 · Limpiar transacciones")
    filas, dup, rech = _tx()
    _df(r, "tx", _CT, filas, "sin duplicados, con tipo, monto y fecha limpios, solo aprobadas y con el índice reiniciado",
        indice=list(range(len(filas))), tol=0.0051)
    _esc(r, "n_duplicados_tx", dup, "cuenta las filas repetidas de `tx_raw`")
    _esc(r, "n_rechazadas", rech, "cuenta las transacciones rechazadas, sin contar las duplicadas")
    r.fin()


def _tx_cli():
    info = {f[0]: (f[4], f[2]) for f in _clientes()}
    filas, _, _ = _tx()
    return [f + list(info.get(f[2], (None, None))) for f in filas]


def check_etapa_4():
    r = _Revision("Etapa 4 · Unir")
    tc = _tx_cli()
    _df(r, "tx_cli", _CT + ["segmento", "distrito"], tc, "todas las transacciones de `tx`, con el segmento y el distrito del cliente", tol=0.0051)
    _esc(r, "n_tx_sin_cliente", sum(f[-1] is None for f in tc), "cuenta las transacciones sin distrito después de unir")
    _df(r, "reclamos", _CR, _reclamos(), "fecha convertida y resuelto como `True`/`False`")
    r.fin()


def check_etapa_5():
    r = _Revision("Etapa 5 · Preguntas de negocio")
    tc = _tx_cli()
    g = defaultdict(float)
    for f in tc:
        if f[5] < 0 and f[7] is not None:
            g[f[7]] += -f[5]
    _ser(r, "gasto_segmento", [round(g[k], 2) for k in sorted(g)], "suma de los egresos por segmento, en positivo, con 2 decimales",
         indice=sorted(g), tol=0.0051)
    filas, _, _ = _tx()
    mes = defaultdict(float)
    for f in filas:
        mes[f[1][:7]] += f[5]
    fines = {"2026-01": "2026-01-31 00:00:00", "2026-02": "2026-02-28 00:00:00", "2026-03": "2026-03-31 00:00:00"}
    _ser(r, "flujo_mensual", [round(mes[k], 2) for k in sorted(mes)], "suma del monto por mes, con 2 decimales",
         indice=[fines[k] for k in sorted(mes)], tol=0.0051)
    segs = sorted({f[7] for f in tc if f[7] is not None})
    canales = sorted({f[4] for f in tc if f[7] is not None})
    tabla = []
    for s in segs:
        sub = [f[4] for f in tc if f[7] == s]
        tabla.append([round(sub.count(c) / len(sub), 3) for c in canales])
    _df(r, "canal_por_segmento", canales, tabla, "la proporción de cada canal dentro de cada segmento, con 3 decimales", indice=segs, tol=0.00051)
    rec = _reclamos()
    conteo = defaultdict(int)
    for f in rec:
        conteo[f[3]] += 1
    top = sorted(conteo, key=lambda k: -conteo[k])[:3]
    _ser(r, "motivos_top3", [conteo[k] for k in top], "los 3 motivos más frecuentes con su cantidad, de mayor a menor", indice=top)
    res = defaultdict(list)
    for f in rec:
        res[f[3]].append(f[4])
    _ser(r, "pct_resueltos_motivo", [round(sum(res[k]) * 100 / len(res[k]), 1) for k in sorted(res)],
         "porcentaje de reclamos resueltos por motivo, con 1 decimal", indice=sorted(res), tol=0.051)
    r.fin()
    r = _Revision("Etapa 5 · Parte B")
    r.predicciones({
        "pred_nan_grupo": "e675bdd897ba87a607b7c344f97a5152cda452c1b1641515ea9436fd35397ada",
    })
    r.fin()


def _tabla_distrito():
    cli = _clientes()
    distrito = {f[0]: f[2] for f in cli}
    n_cli, n_rec, egr = defaultdict(int), defaultdict(int), defaultdict(float)
    for f in cli:
        n_cli[f[2]] += 1
    for f in _reclamos():
        n_rec[distrito[f[1]]] += 1
    for f in _tx_cli():
        if f[5] < 0 and f[8] is not None:
            egr[f[8]] += -f[5]
    filas = {d: [n_cli[d], n_rec[d], round(n_rec[d] * 100 / n_cli[d], 1), round(egr[d], 2)] for d in n_cli}
    orden = sorted(filas, key=lambda d: (-filas[d][2], d))
    return orden, filas, max(n_rec, key=n_rec.get)


def check_reto():
    r = _Revision("Reto final")
    orden, filas, bruto = _tabla_distrito()
    _df(r, "tabla_distrito", ["n_clientes", "n_reclamos", "reclamos_por_100", "egresos"], [filas[d] for d in orden],
        "una fila por distrito, ordenadas de mayor a menor `reclamos_por_100`", indice=orden, tol=0.051)
    es_texto = lambda a, b: isinstance(a, str) and a == b
    r.valor("distrito_mas_reclamos", bruto, None, "el distrito con más reclamos en números absolutos", igual=es_texto)
    r.valor("distrito_mas_reclamos_relativo", orden[0], None, "el distrito con más reclamos por cada 100 clientes", igual=es_texto)
    r.fin()


def check_pro():
    r = _Revision("Nivel pro")
    cli = _clientes()
    con_reclamo = {f[1] for f in _reclamos()}
    ingresos = [f[5] for f in cli]
    q = statistics.quantiles(ingresos, n=4, method="inclusive")
    bordes = [min(ingresos) - 1] + q + [max(ingresos)]
    grupos = defaultdict(list)
    for f in cli:
        etiqueta = next(e for a, b, e in zip(bordes, bordes[1:], ["Q1", "Q2", "Q3", "Q4"]) if a < f[5] <= b)
        grupos[etiqueta].append(f[0] in con_reclamo)
    _ser(r, "tasa_reclamo_cuartil", [round(sum(grupos[k]) / len(grupos[k]), 3) for k in ["Q1", "Q2", "Q3", "Q4"]],
         "proporción de clientes con al menos un reclamo en cada cuartil de ingreso, con 3 decimales", indice=["Q1", "Q2", "Q3", "Q4"], tol=0.00051)
    r.fin()


print("✅ Setup listo. Archivos del caso escritos y verificadores cargados.")

### 📦 Los archivos del caso
Los datos se generan con una semilla fija, así que siempre salen iguales. Mira las primeras líneas de cada archivo antes de empezar: fíjate en los separadores, los formatos de fecha y los valores raros.

In [ ]:
for archivo in ["clientes_banco.csv", "transacciones_2026_01.csv", "transacciones_2026_02.csv",
                "transacciones_2026_03.csv", "reclamos.csv"]:
    with open(archivo, encoding="utf-8") as f:
        print(f"--- {archivo}")
        for _ in range(4):
            print(f.readline().rstrip())

---
## Etapa 1 · Cargar

### 📘 Repaso
`pd.read_csv(ruta, sep=...)` carga un archivo; `pd.concat([...], ignore_index=True)` apila varios con un índice nuevo. Una comprensión de lista carga varios archivos con nombres parecidos en una línea.

In [ ]:
for mes in [1, 2, 3]:
    print(f"transacciones_2026_{mes:02d}.csv")       # los nombres de los tres archivos

### ✍️ Tu turno · Etapa 1
1. `clientes_raw`: `clientes_banco.csv` tal cual.
2. `tx_raw`: los tres archivos de transacciones apilados en orden (enero, febrero, marzo), con un índice nuevo.
3. `reclamos_raw`: `reclamos.csv` tal cual.
4. `forma_tx`: la forma de `tx_raw`.

Explora las tres tablas con `head`, `info` y `describe` antes de seguir.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_etapa_1()

<details><summary>💡 Pista 1</summary>

Las transacciones usan punto y coma. Puedes cargar cada mes en una variable y después apilarlas.
</details>

<details><summary>💡 Pista 2</summary>

En una línea: `pd.concat([pd.read_csv(f"transacciones_2026_{m:02d}.csv", sep=";") for m in [1, 2, 3]], ignore_index=True)`.
</details>

---
## Etapa 2 · Limpiar clientes

### 📘 Repaso
- `na_values=["unknown"]` al leer, o `.replace(valor, np.nan)` después, para convertir marcadores en nulos.
- `.str.strip().str.title()` para estandarizar textos.
- `pd.to_datetime(col, format="%d/%m/%Y")` para fechas día/mes/año.
- `nunique()` cuenta valores distintos.

In [ ]:
distritos_ej = pd.Series([" surco", "SURCO ", "Surco", "san isidro"])
print(distritos_ej.nunique(), distritos_ej.str.strip().str.title().nunique())

### ✍️ Tu turno · Etapa 2
Crea `clientes`, con las mismas columnas que `clientes_raw`, pero:
1. `segmento`: `"unknown"` convertido en nulo.
2. `fecha_alta`: convertida a fecha (está como día/mes/año).
3. `distrito`: sin espacios a los lados y con mayúscula inicial en cada palabra.
4. `edad`: el centinela `-1` convertido en nulo.

Después, `n_distritos`: cuántos distritos distintos quedan. `clientes_raw` no debe cambiar.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_etapa_2()

<details><summary>💡 Pista 1</summary>

Puedes volver a leer el archivo con `na_values=["unknown"]` y luego arreglar las otras tres columnas, una línea cada una.
</details>

<details><summary>💡 Pista 2</summary>

Compara `clientes_raw["distrito"].nunique()` con `clientes["distrito"].nunique()`: si la limpieza funcionó, el segundo número es mucho menor.
</details>

---
## Etapa 3 · Limpiar transacciones

### 📘 Repaso
- `df.duplicated().sum()` cuenta filas repetidas y `drop_duplicates()` las quita.
- Un monto escrito como texto se limpia con `.str.replace` y se convierte con `astype(float)`.
- Si vas a modificar columnas de una tabla que salió de filtrar o de `drop_duplicates`, agrega `.copy()`: así trabajas sobre una tabla independiente y pandas no muestra el aviso `SettingWithCopyWarning`.
- Después de filtrar filas, el índice queda con huecos. `reset_index(drop=True)` lo renumera de 0 en adelante (el `drop=True` evita guardar el índice viejo como columna).

In [ ]:
filtrado_ej = pd.DataFrame({"x": [10, 20, 30, 40]}).query("x > 15")
print(filtrado_ej.index.tolist(), filtrado_ej.reset_index(drop=True).index.tolist())

### ✍️ Tu turno · Etapa 3
1. `n_duplicados_tx`: cuántas filas repetidas tiene `tx_raw`.
2. `tx`: a partir de `tx_raw`, en este orden:
   - quita las filas duplicadas;
   - `tipo` sin espacios a los lados y en minúsculas;
   - `monto` como número (quita `"S/"`, las comas de miles y los espacios);
   - `fecha` convertida a fecha;
   - deja solo las transacciones con estado `"aprobada"`;
   - reinicia el índice.
3. `n_rechazadas`: cuántas transacciones rechazadas había (sin contar duplicadas).

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_etapa_3()

<details><summary>💡 Pista 1</summary>

Calcula `n_rechazadas` después de quitar duplicados y antes de filtrar las aprobadas.
</details>

<details><summary>💡 Pista 2</summary>

Un buen esquema: `tx = tx_raw.drop_duplicates().copy()` (el `.copy()` evita un aviso de pandas al modificar columnas), luego `tx["tipo"] = ...`, `tx["monto"] = ...`, `tx["fecha"] = ...`, y al final `tx = tx[tx["estado"] == "aprobada"].reset_index(drop=True)`.
</details>

---
## Etapa 4 · Unir

### 📘 Repaso
`izq.merge(der, on="clave", how="left")` conserva todas las filas de la izquierda. Selecciona antes solo las columnas que necesitas de la tabla derecha: `der[["clave", "col1", "col2"]]`. Cuenta las filas antes y después: con una unión `left` y claves únicas a la derecha, deben ser las mismas.

In [ ]:
ventas_ej = pd.DataFrame({"cliente": ["A", "B", "Z"], "monto": [10, 20, 30]})
datos_ej = pd.DataFrame({"cliente": ["A", "B"], "ciudad": ["Lima", "Cusco"], "telefono": ["...", "..."]})
unido_ej = ventas_ej.merge(datos_ej[["cliente", "ciudad"]], on="cliente", how="left")
print(unido_ej, len(ventas_ej) == len(unido_ej))

### ✍️ Tu turno · Etapa 4
1. `tx_cli`: todas las transacciones de `tx` con el `segmento` y el `distrito` del cliente (en ese orden), desde `clientes`.
2. `n_tx_sin_cliente`: cuántas transacciones quedaron sin distrito (clientes que no están en la base).
3. `reclamos`: una copia de `reclamos_raw` con la `fecha` convertida a fecha y `resuelto` convertido en `True` (`"Sí"`) o `False` (`"No"`) con `map`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_etapa_4()

<details><summary>💡 Pista 1</summary>

`clientes[["cliente_id", "segmento", "distrito"]]` es la tabla derecha. Los nulos se cuentan con `.isna().sum()`.
</details>

<details><summary>💡 Pista 2</summary>

Para `resuelto`: `reclamos["resuelto"].map({"Sí": True, "No": False})`.
</details>

---
## Etapa 5 · Preguntas de negocio

### 📘 Repaso
`groupby` + función, `crosstab(..., normalize="index")`, `value_counts().head(n)` y `set_index("fecha").resample("ME")`. Por defecto, `groupby` **deja fuera** las filas cuyo grupo es nulo.

In [ ]:
d_ej = pd.DataFrame({"g": ["a", None, "a", "b"], "x": [1, 2, 3, 4]})
print(d_ej.groupby("g")["x"].sum())       # la fila con grupo nulo no aparece

### ✍️ Tu turno · Etapa 5
**Parte A.**
1. `gasto_segmento`: la suma de los egresos (montos negativos) de `tx_cli` por segmento, **en positivo**, con 2 decimales.
2. `flujo_mensual`: la suma del monto de `tx` por mes, con 2 decimales.
3. `canal_por_segmento`: la proporción de transacciones de cada canal dentro de cada segmento (filas: segmento), con 3 decimales.
4. `motivos_top3`: los 3 motivos de reclamo más frecuentes, con su cantidad.
5. `pct_resueltos_motivo`: el porcentaje de reclamos resueltos por motivo, con 1 decimal.

**Parte B.** ¿Aparecen en `gasto_segmento` los clientes con segmento nulo? Responde en `pred_nan_grupo` con `"sí"` o `"no"`.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_etapa_5()

<details><summary>💡 Pista 1</summary>

Para los egresos, filtra `tx_cli["monto"] < 0` antes de agrupar y usa `.abs()` al final. Como `resuelto` es `True`/`False`, su promedio ya es la proporción de resueltos.
</details>

<details><summary>💡 Pista 2</summary>

`flujo_mensual = tx.set_index("fecha")["monto"].resample("ME").sum().round(2)`. Los demás siguen los patrones de la sesión 11.
</details>

---
## 🏋️ Reto final: ¿dónde está el problema de reclamos?
Gerencia quiere saber qué distrito tiene "más reclamos". Contar reclamos engaña: el distrito con más clientes siempre tendrá más reclamos. La comparación justa es **por cada 100 clientes**.

Construye `tabla_distrito`, con el distrito como índice y estas columnas, en este orden:
- `n_clientes`: cuántos clientes tiene el distrito en `clientes`.
- `n_reclamos`: cuántos reclamos hicieron sus clientes (0 si ninguno).
- `reclamos_por_100`: `n_reclamos / n_clientes * 100`, con 1 decimal.
- `egresos`: la suma de los egresos de `tx_cli` de sus clientes, en positivo, con 2 decimales.

Ordénala de mayor a menor `reclamos_por_100`. Luego:
- `distrito_mas_reclamos`: el distrito con más reclamos en números absolutos.
- `distrito_mas_reclamos_relativo`: el distrito con más reclamos por cada 100 clientes.

¿Coinciden? Escribe en una celda de texto qué le dirías a gerencia en dos líneas.

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_reto()

<details><summary>💡 Pista 1</summary>

Arma tres Series con el distrito como índice (clientes por distrito, reclamos por distrito y egresos por distrito) y júntalas en un DataFrame. Para saber el distrito de cada reclamo, une `reclamos` con `clientes`.
</details>

<details><summary>💡 Pista 2</summary>

`pd.DataFrame({"n_clientes": s1, "n_reclamos": s2, "egresos": s3})` alinea las tres por índice. Rellena con `fillna(0)` los distritos sin reclamos, calcula la columna normalizada, reordena las columnas y usa `sort_values`.
</details>

---
## 🚀 Nivel pro (opcional)
`tasa_reclamo_cuartil`: divide a los clientes en cuartiles de `ingreso_mensual` con `pd.qcut` (etiquetas `"Q1"` a `"Q4"`) y calcula, en cada cuartil, qué proporción de clientes hizo al menos un reclamo, con 3 decimales. ¿Reclaman más los clientes de mayores ingresos?

In [ ]:
# Tu código aquí


In [ ]:
# ✅ Verificar
check_pro()

---
## 🧱 Avance del proyecto: P2 · Limpieza y tablas de análisis

Aplica a tu proyecto exactamente el flujo de hoy. Trabaja en `notebooks/02_limpieza_unificacion.ipynb`.

**Qué hacer**
1. **Unificar**: apila los archivos de todos los años con `concat`, después de alinear los nombres de las columnas que cambian entre años (anotados en la parte 1 de P2). Agrega una columna `anio` si el archivo no la trae.
2. **Normalizar nombres de entidades**: una misma entidad suele aparecer escrita de varias formas (mayúsculas, siglas, "S.A." o "S.A.A.", tildes). Estandariza con `.str.strip().str.upper()` y un diccionario de equivalencias aplicado con `replace` o `map`. Revisa el resultado con `value_counts()`.
3. **Tipos, nulos y centinelas**: convierte fechas y números, trata los valores centinela que encontraste en P1 y decide qué hacer con cada columna con nulos (eliminar filas, rellenar o dejar como "sin dato"). Anota cada decisión.
4. **Datos personales**: elimina las columnas que identifican personas.
5. **Guardar**: exporta la tabla limpia con `to_csv` en una carpeta local (por ejemplo `data/processed/`) que no se sube al repositorio si pesa mucho; documenta en `data/README.md` cómo regenerarla.
6. **Tablas de análisis** para las preguntas 1 a 3 del proyecto: denuncias por entidad y año (`pivot_table`), por producto y por motivo (`value_counts`), y por región (`groupby`). Si ya tienes datos de tamaño de las entidades (clientes o créditos de la SBS), prepara también la tabla **normalizada**, como `reclamos_por_100` del reto de hoy.

**Por qué lo haría un analista**
La limpieza decide qué tan confiables son las conclusiones: una entidad escrita de tres formas aparece como tres entidades pequeñas, y un ranking sin normalizar solo refleja el tamaño. Documentar cada decisión permite defender los números cuando alguien los cuestione.

**Cómo debe verse el resultado**
Un notebook que va del archivo crudo a una tabla limpia sin pasos manuales, con celdas de texto que explican cada decisión y muestran el antes y el después (filas, nulos y valores distintos por columna), y al final las tablas de análisis listas para graficar en el módulo 4.

---
## ✅ Cierre: autoevaluación
Marca lo que puedes hacer sin mirar:
- [ ] Cargar y apilar varios archivos con una comprensión de lista.
- [ ] Limpiar una tabla real: textos, fechas, centinelas, "unknown" y duplicados.
- [ ] Reiniciar el índice después de filtrar y explicar para qué sirve `drop=True`.
- [ ] Unir tablas con `merge` sin perder ni duplicar filas, y verificarlo.
- [ ] Explicar qué pasa con los grupos nulos en `groupby`.
- [ ] Explicar por qué contar reclamos engaña y calcular una tasa por cada 100 clientes.
- [ ] Contar en dos líneas, para alguien que no programa, lo que encontré.

**Próxima sesión (S14):** empezamos Matplotlib: la anatomía de un gráfico.